In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window 
from delta.tables import DeltaTable
from datetime import date

df_filter = spark.read.table('futures.staging_yf').filter(col('SI_F_Close') > 50)
#df_filter = df_filter.withColumn("datekey", date_format(col("Date_"), 'yyyyMMdd'))
df = spark.read.table('futures.staging_yf').filter(col('SI_F_Close')<=50)
#df_jn = df_filter.alias("high").join(df.alias("low"), col('high.Date_') == col('low.Date_'), how='left')
df_union = df_filter.union(df)
display(df_union.show())
df_union.write.mode("overwrite").saveAsTable("futures.union_tbl")


#create join on date dimension
import pyspark.sql.functions as f
from datetime import date
from delta.tables import DeltaTable
df_bls = spark.read.table('futures.bls').filter((year(col('Dt')).isin(2024, 2025))& (col('Series_Nm')=='Labor'))\
    .select('Series_Nm', 'Dt', date_format(col('Dt'), 'yyyyMMdd'), 'Val')
df_bls = df_bls.withColumnRenamed("date_format(Dt, yyyyMMdd)", "datekey")

df_slv = spark.read.table('futures.staging_yf').filter(year(col("Date_")).isin(2024,2025))
df_slv_bls = df_slv.alias('slv').join(df_bls.alias('bls'), "bls.Dt ==slv.Date_",how='')
df_jn = df_slv.toPandas()
df_jn.head(40)







start_dt = date(2010,1,1)
end_dt = date(2025, 1,1)
date_df = spark.sql(f"SELECT explode(sequence(to_date('{start_dt}'), to_date('{end_dt}'), interval 1 day)) AS calendar_date")
display(date_df.show())
date_dimension_df = date_df.withColumn("date_key", date_format(col("calendar_date"), "yyyyMMdd").cast("integer")) \
        .withColumn("day_of_month", dayofmonth(col("calendar_date"))) \
        .withColumn("day_of_week", dayofweek(col("calendar_date"))) \
        .withColumn("day_of_year", dayofyear(col("calendar_date"))) \
        .withColumn("week_of_year", weekofyear(col("calendar_date"))) \
        .withColumn("month", month(col("calendar_date"))) \
        .withColumn("month_name", date_format(col("calendar_date"), "MMMM")) \
        .withColumn("quarter", quarter(col("calendar_date"))) \
        .withColumn("year", year(col("calendar_date"))) \
        .withColumn("is_weekend", when(dayofweek(col("calendar_date")).isin(1, 7), lit(True)).otherwise(False))
display(date_dimension_df.show())
date_dimension_df.write.mode("overwrite").saveAsTable("futures.dim_date")

,Date_,SI_F_Open,SI_F_High,SI_F_Low,SI_F_Close,SI_F_Volume,GC_F_Open,GC_F_High,GC_F_Low,GC_F_Close,GC_F_Volume,exe_timeStamp
0,2024-01-02,23.844999,24.070000,23.733000,23.733000,20,2063.500000,2073.699951,2057.100098,2064.399902,61,2025-11-21 00:03:03.241326
1,2024-01-03,23.264999,23.264999,22.930000,22.945999,458,2034.199951,2044.000000,2034.199951,2034.199951,54,2025-11-21 00:03:03.241326
2,2024-01-04,22.930000,23.045000,22.705000,22.989000,24,2041.599976,2044.500000,2038.000000,2042.300049,88,2025-11-21 00:03:03.241326
3,2024-01-05,23.065001,23.122000,22.865000,23.122000,5,2044.500000,2048.100098,2042.400024,2042.400024,12,2025-11-21 00:03:03.241326
4,2024-01-08,22.990000,23.120001,22.834999,23.120001,53,2019.099976,2033.699951,2019.099976,2026.599976,10,2025-11-21 00:03:03.241326
5,2024-01-09,23.139999,23.139999,22.903999,22.903999,19,2035.800049,2035.800049,2026.400024,2026.400024,27,2025-11-21 00:03:03.241326
6,2024-01-10,23.000000,23.000000,22.799999,22.884001,10,2029.000000,2035.599976,2021.699951,2021.699951,538,2025-11-21 00:03:03.241326
7,2024-01-11,22.959999,23.084999,22.535000,22.537001,109,2025.099976,2025.099976,2014.300049,2014.300049,97,2025-11-21 00:03:03.241326
8,2024-01-12,22.825001,23.379999,22.825001,23.162001,44,2031.099976,2057.000000,2031.099976,2046.699951,390,2025-11-21 00:03:03.241326
9,2024-01-16,22.933001,22.933001,22.933001,22.933001,0,2051.699951,2054.800049,2026.000000,2026.000000,46,2025-11-21 00:03:03.241326


In [0]:
%sql
USE futures;
select Series_ID, Dt, round(Per_Change, 2) from futures.bls where Series_Nm rlike '^L*';
DROP TABLE IF EXISTS futures.tbl;
--CREATE TABLE TBL AS SELECT * FROM futures.bls;
create temporary view tbl as  select * from futures.bls where year(dt)=2025;

;with x as 
(select Series_ID, Dt, round(Per_Change, 2) from futures.bls where Series_Nm rlike '^L*'
);create temporary v


describe extended futures.tbl;




---------------------------------------------------------------------------
ParseException                            Traceback (most recent call last)
File <command-6987474304140984>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "USE futures;\nselect Series_ID, Dt, round(Per_Change, 2) from futures.bls where Series_Nm rlike '^L*';\nDROP TABLE IF EXISTS futures.tbl;\n--CREATE TABLE TBL AS SELECT * FROM futures.bls;\ncreate temporary view tbl as  select * from futures.bls where year(dt)=2025;\n\n;with x as \n(select Series_ID, Dt, round(Per_Change, 2) from futures.bls where Series_Nm rlike '^L*'\n);create temporary v\n\n\ndescribe extended futures.tbl;\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being 

In [0]:

from pyspark.sql.functions import *
path = "abfss://bronze@greedystore.dfs.core.windows.net/"
file = "Commodity_2020.csv"
outfil = "Commodit_out"
df = spark.read.format("csv").option("header","true")\
    .option("inferSchmea", "true").load(f"{path}{file}").toDF(*[col.replace('=','_') for col in df.columns])
display(df)
df.write.mode("append").csv(f"{path}{outfil}")




Dt,PL_F_Open,PL_F_High,PL_F_Low,PL_F_Close,PL_F_Volume,SI_F_Open,SI_F_High,SI_F_Low,SI_Close,SI_Vol,GC_F_Open,GC_F_High,GC_F_Low,GC_Close,GC_Vol,HG_F_Open,HG_F_High,HG_F_Low,HG_F_Close,HG_F_Volume,DXY_Open,DXY_High,DXY_Low,DXY_Close,DXY_Volume,PA_F_Open,PA_F_High,PA_F_Low,PA_F_Close,PA_F_Volume,datekey
1/3/2023,1078.699951,1092.900024,1074.5,1082,156,24.45499992,24.53000069,23.97999954,24.05900002,148,1836.199951,1839.699951,1836.199951,1839.699951,29,3.832499981,3.842000008,3.761499882,3.766000032,1032,NULL,NULL,NULL,NULL,NULL,1684.199951,1684.199951,1684.199951,1684.199951,0,20230103
1/4/2023,1100,1100.5,1080.699951,1080.699951,249,24.08499908,24.10499954,23.75,23.79199982,24,1845.599976,1859.099976,1845.599976,1852.800049,25,3.736000061,3.752000093,3.727499962,3.739500046,663,NULL,NULL,NULL,NULL,NULL,1788,1788,1788,1788,0,20230104
1/5/2023,1079,1079,1059.099976,1059.099976,10,23.5,23.5,23.14500046,23.25799942,8,1855.199951,1855.199951,1834.800049,1834.800049,24,3.800499916,3.818000078,3.800499916,3.818000078,367,NULL,NULL,NULL,NULL,NULL,1724,1724,1724,1724,0,20230105
1/6/2023,1063.800049,1096.400024,1063.800049,1092.599976,95,23.29500008,23.8220005,23.29000092,23.8220005,6,1838.400024,1868.199951,1835.300049,1864.199951,26,3.813499928,3.913500071,3.813499928,3.908499956,422,NULL,NULL,NULL,NULL,NULL,1798,1798,1798,1798,2,20230106
1/9/2023,1093.699951,1093.699951,1086.300049,1086.300049,251,23.71199989,23.71199989,23.71199989,23.71199989,2,1867,1880,1867,1872.699951,62,3.943500042,4.02699995,3.938499928,4.018499851,837,NULL,NULL,NULL,NULL,NULL,1766.699951,1768,1748.5,1766.699951,2,20230109
1/10/2023,1078,1078,1074.5,1076.599976,32,23.43000031,23.50699997,23.42499924,23.50699997,4,1877.800049,1878.099976,1871.599976,1871.599976,101,4.012499809,4.066999912,4.008500099,4.066999912,676,NULL,NULL,NULL,NULL,NULL,1768.300049,1768.300049,1768.300049,1768.300049,2,20230110
1/11/2023,1080,1091,1073.800049,1073.800049,69,23.54999924,23.80500031,23.18499947,23.32699966,9,1873.099976,1877.800049,1873.099976,1874.599976,1236,4.108500004,4.173999786,4.105999947,4.154500008,657,NULL,NULL,NULL,NULL,NULL,1771.300049,1771.300049,1771.300049,1771.300049,2,20230111
1/12/2023,1068,1074.599976,1064,1074.599976,2,23.80500031,23.86300087,23.80500031,23.86300087,33,1877.199951,1897.400024,1877.199951,1895.5,59,4.150499821,4.190999985,4.145999908,4.18599987,532,NULL,NULL,NULL,NULL,NULL,1781.099976,1781.099976,1781.099976,1781.099976,2,20230112
1/13/2023,1067.300049,1067.300049,1049,1063.699951,18,23.88999939,24.2310009,23.5,24.2310009,30,1899,1920.900024,1898.300049,1918.400024,512,4.178999901,4.207499981,4.146999836,4.207499981,372,NULL,NULL,NULL,NULL,NULL,1777.5,1777.5,1777.5,1777.5,2,20230113
1/17/2023,1056,1056,1037.5,1037.5,8,24.30999947,24.31500053,23.94400024,23.94400024,118,1920.099976,1920.099976,1905.199951,1907.199951,706,4.140999794,4.219500065,4.140999794,4.215000153,406,NULL,NULL,NULL,NULL,NULL,1724.199951,1724.199951,1724.199951,1724.199951,2,20230117


In [0]:

from pyspark.sql.functions import *
path = "abfss://bronze@greedystore.dfs.core.windows.net/Commodit_out/"
files = [file for file in dbutils.fs.ls(path) if file.name.endswith(".csv")]
display(files)
file_names = spark.createDataFrame(files)
display(file_names.show())
display(max(file_names.modificationTime))
file_names.orderBy("modificationTime", desc=True).show()
file_names = file_names.withColumn("time_exe", (col("modificationTime")/1000).cast("timestamp"))
display(file_names.show())


path = "abfss://bronze@greedystore.dfs.core.windows.net/"
file = [files for files in dbutils.fs.ls(path)]

display(file)

path,name,size,modificationTime
abfss://bronze@greedystore.dfs.core.windows.net/BLS_ADF2025-11-17 01:20:20-00001.csv,BLS_ADF2025-11-17 01:20:20-00001.csv,26002,1763342426000
abfss://bronze@greedystore.dfs.core.windows.net/Commodit_out/,Commodit_out/,0,1763856374000
abfss://bronze@greedystore.dfs.core.windows.net/Commodity_2020.csv,Commodity_2020.csv,201053,1763250805000
abfss://bronze@greedystore.dfs.core.windows.net/Commodity_Parquet/,Commodity_Parquet/,0,1763250886000
abfss://bronze@greedystore.dfs.core.windows.net/delta_store/,delta_store/,0,1763863602000
